In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
class DenseLayer:
    def __init__(self, no_of_features: int, no_of_neurons: int, activation: str = 'sigmoid'):
        self.w = np.random.randn(no_of_features, no_of_neurons) * np.sqrt(2.0 / no_of_features)
        self.b = np.zeros((1, no_of_neurons))
        self.activation = activation.lower()
        
        self.x = None
        self.z = None
        self.a = None

    def _activate(self, z):
        if self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        elif self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'tanh':
            return np.tanh(z)
        elif self.activation == 'softmax':
            exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
            return exp_z / np.sum(exp_z, axis=-1, keepdims=True)
        else:
            raise ValueError(f"Activation '{self.activation}' is not supported.")

    def _activate_derivative(self, da):
        if self.activation == 'sigmoid':
            sig = self.a
            return da * sig * (1 - sig)
        elif self.activation == 'relu':
            return da * (self.z > 0)
        elif self.activation == 'tanh':
            return da * (1 - np.square(self.a))
        elif self.activation == 'softmax':
            return da 
        return da

    def forward(self, x):
        self.x = x
        self.z = np.dot(x, self.w) + self.b
        self.a = self._activate(self.z)
        return self.a

    def backward(self, da, learning_rate: float):
        dz = self._activate_derivative(da)
        
        m = self.x.shape[0]
        dw = np.dot(self.x.T, dz) / m
        db = np.sum(dz, axis=0, keepdims=True) / m
        
        dx = np.dot(dz, self.w.T)
        
        self.w -= learning_rate * dw
        self.b -= learning_rate * db
        
        return dx

In [ ]:
class ANN:
    def __init__(self):
        self.layers = []

    def add_layer(self, no_of_features: int, no_of_neurons: int, activation: str = 'sigmoid'):
        self.layers.append(DenseLayer(no_of_features, no_of_neurons, activation))

    def forward(self, x):
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, y_true, y_pred, learning_rate: float = 0.01):
        da = 2 * (y_pred - y_true)
        
        for layer in reversed(self.layers):
            da = layer.backward(da, learning_rate)

    def train_step(self, x, y, learning_rate: float = 0.01):
        y_pred = self.forward(x)
        loss = np.mean(np.square(y_pred - y))
        self.backward(y, y_pred, learning_rate)
        return loss